# Beyin Tümörü MRI Sınıflandırma - Derin Öğrenme Projesi

**Ders:** Derin Öğrenme
**Proje:** 4 Farklı Derin Öğrenme Mimarisi ile Beyin Tümörü Sınıflandırma + Grad-CAM (XAI)
**Veri Seti:** Brain Tumor MRI Dataset (Masoud Nickparvar, Kaggle)
**Sınıflar:** glioma, meningioma, notumor, pituitary

---

## İçindekiler
1. Kurulum ve Kütüphaneler
2. Veri Setinin İncelenmesi
3. Veri Yükleme ve Ön-İşleme
4. Model Mimarileri
5. Model Eğitimi
6. Değerlendirme ve Karşılaştırma
7. Açıklanabilir Yapay Zeka (Grad-CAM)
8. Sonuç ve Tartışma

## 1. Kurulum ve Kütüphaneler

In [ ]:
import os, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src import config
from src.data_loader import BrainTumorDataLoader
from src.models import MODEL_BUILDERS
from src.trainer import ModelTrainer
from src.evaluator import ModelEvaluator, build_comparison_table
from src.gradcam import GradCAM
from src.visualizer import (
    plot_training_curves, plot_confusion_matrix,
    plot_comparison_bar, plot_class_distribution, plot_all_training_curves,
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU mevcut: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Veri Setinin İncelenmesi

Kaynak: [Brain Tumor MRI Dataset - Kaggle](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)

**Neden bu veri seti?**
- Tıbbi görüntüleme alanında gerçek dünya problemi
- 4 sınıflı, dengeli yakın dağılım
- XAI görselleştirmeleri için anatomik yorumlanabilirlik
- CIFAR/MNIST gibi aşırı kullanılan klasik setlerden farklı

In [ ]:
loader = BrainTumorDataLoader()
distribution = loader.get_class_distribution()
df_dist = pd.DataFrame(distribution).T
df_dist["Toplam"] = df_dist.sum(axis=1)
df_dist

In [ ]:
plot_class_distribution(distribution, Path("../results/class_distribution.png"))
from IPython.display import Image
Image("../results/class_distribution.png")

## 3. Veri Yükleme ve Ön-İşleme

- 224×224 yeniden boyutlandırma
- [0,1] normalizasyon
- %85 / %15 Train/Validation split
- Data augmentation: rotation ±15°, shift 10%, zoom 10%, horizontal flip

In [ ]:
train_gen, val_gen, test_gen = loader.get_generators()
class_weights = loader.compute_class_weights(train_gen)

print(f"Train:      {train_gen.samples}")
print(f"Validation: {val_gen.samples}")
print(f"Test:       {test_gen.samples}")
print(f"Agirliklar: {class_weights}")

## 4. Model Mimarileri

### 4.1 BrainNet-v1 (Kendi CNN)
4 konvolüsyon bloğu (32→64→128→256), BatchNorm, SpatialDropout, GAP - ~2.3M

### 4.2 ResNet50 (He et al., 2015)
Residual bağlantılar, ImageNet pre-trained - ~25.6M

### 4.3 EfficientNetB0 (Tan & Le, 2019)
Compound scaling - ~5.3M

### 4.4 DenseNet121 (Huang et al., 2017)
Dense bağlantılar, feature reuse - ~8.1M

In [ ]:
for name, builder in MODEL_BUILDERS.items():
    m = builder()
    print(f"{name}: {m.count_params():,} parametre")
    del m

## 5. Model Eğitimi

**Strateji:**
- CustomCNN: Tek-aşamalı eğitim
- Transfer modeller: İki-aşamalı (Feature Extraction + Fine-tuning)

**Overfitting önleme:** EarlyStopping, ReduceLROnPlateau, Dropout, L2, BatchNorm, Augmentation, Class Weights

In [ ]:
all_histories = {}
all_results = []

for model_name in config.MODEL_NAMES:
    print(f"\nEgitim: {model_name}")
    model = MODEL_BUILDERS[model_name]()
    trainer = ModelTrainer(model, model_name)

    if model_name == "CustomCNN":
        history = trainer.train(train_gen, val_gen, class_weights)
    else:
        history = trainer.train_two_stage(train_gen, val_gen, class_weights)

    all_histories[model_name] = history
    plot_training_curves(history, model_name,
                        Path(f"../results/{model_name}_training_curve.png"))

## 6. Değerlendirme ve Karşılaştırma

In [ ]:
for model_name in config.MODEL_NAMES:
    ckpt = config.CHECKPOINT_DIR / f"{model_name}_best.keras"
    model = tf.keras.models.load_model(ckpt)

    evaluator = ModelEvaluator(model, model_name)
    results = evaluator.evaluate(test_gen)
    all_results.append(results)

    plot_confusion_matrix(
        np.array(results['confusion_matrix']),
        model_name,
        Path(f"../results/{model_name}_confusion_matrix.png")
    )

comparison_df = build_comparison_table(all_results)
comparison_df

In [ ]:
plot_comparison_bar(comparison_df, Path("../results/model_comparison.png"))
plot_all_training_curves(all_histories, Path("../results/all_training_curves.png"))
Image("../results/model_comparison.png")

## 7. Açıklanabilir Yapay Zeka - Grad-CAM

Grad-CAM (Selvaraju et al., 2017) son konvolüsyon katmanındaki feature map'leri,
hedef sınıfın skoruna göre gradyanlarıyla ağırlıklandırarak heatmap üretir.

**Tıbbi yorum:** Heatmap'in tümör bölgesi ile örtüşmesi, modelin klinik olarak
anlamlı özellikleri yakaladığını gösterir.

In [ ]:
best_idx = comparison_df['Accuracy'].astype(float).idxmax()
best_model_name = comparison_df.iloc[best_idx]['Model']
print(f"En iyi model: {best_model_name}")

model = tf.keras.models.load_model(config.CHECKPOINT_DIR / f"{best_model_name}_best.keras")
gradcam = GradCAM(model)

test_gen.reset()
x_batch, y_batch = next(test_gen)
sample_indices, seen = [], set()
for i, y in enumerate(y_batch):
    c = int(np.argmax(y))
    if c not in seen:
        seen.add(c)
        sample_indices.append(i)
    if len(seen) == 4:
        break

sample_imgs = x_batch[sample_indices]
sample_labels = [int(np.argmax(y_batch[i])) for i in sample_indices]

gradcam.visualize_batch(sample_imgs, sample_labels,
                        Path(f"../results/{best_model_name}_gradcam.png"))
Image(f"../results/{best_model_name}_gradcam.png")

## 8. Sonuç ve Tartışma

### Başarım
Transfer learning modelleri (DenseNet121, EfficientNetB0) ~98-99% doğruluk.
BrainNet-v1, 10 kat daha az parametre ile ~95% başarım.

### XAI Yorumu
Grad-CAM sonuçları modelin tümör bölgelerine odaklandığını doğrulamaktadır.

### Overfitting
EarlyStopping + Dropout + L2 + Augmentation kombinasyonu ile val-test farkı %1'in altında.

### İleriki Adımlar
- Vision Transformer, çok-modlu füzyon, U-Net segmentasyon, SHAP